<a href="https://colab.research.google.com/github/kny1209/test2/blob/main/AI/DR-08079_%EA%B5%AC%EB%82%98%EC%98%81_AI%EC%9D%91%EC%9A%A9_11%EC%B0%A8%EC%8B%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 문제 1. Faster R-CNN의 속도 혁신
R-CNN과 Fast R-CNN은 후보 영역(Region Proposal)을 추출하기 위해 CPU 기반의 'Selective Search' 알고리즘을 사용했습니다. 이는 전체 추론 속도의 병목이었습니다. **Faster R-CNN**은 이를 해결하기 위해 **GPU** 상에서 동작하며 학습이 가능한 **이 네트워크**를 도입하여 End-to-End 학습을 가능하게 했습니다. 이 네트워크의 이름은 무엇입니까?

**답안:** RPN (학습 가능한 신경망)

### 문제 2. 객체 검출 평가 기준 (Metric)
객체 검출 모델의 성능을 평가할 때 PASCAL VOC 데이터셋은 IoU 0.5 이상을 정답으로 간주합니다. 반면, 더 난이도가 높고 엄격한 **MS COCO** 데이터셋은 IoU 임계값(Threshold)을 어떻게 설정하여 mAP를 계산하는지 서술하시오.

**답안:** mAp@[0.5:0.95]

### 문제 3. IoU (Intersection over Union) 계산 함수 구현
두 개의 Bounding Box가 주어졌을 때, 교집합(Intersection)과 합집합(Union)을 이용하여 IoU를 계산하는 함수를 완성하시오.
(입력 박스 포맷: `[x1, y1, x2, y2]`)

In [3]:
import numpy as np

def compute_iou(boxA, boxB):
    # 1. 교집합 영역(Intersection)의 좌표 계산
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    # 2. 교집합의 넓이 (겹치지 않는 경우 0 처리)
    inter_w = max(0, xB - xA)
    inter_h = max(0, yB - yA)
    inter_area = max(xB - xA, 0) * max(yB - yA, 0)

    # 3. 각 박스의 넓이 계산
    boxA_area = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxB_area = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    # 4. 합집합(Union) 넓이 계산
    union_area = boxA_area + boxB_area - inter_area

    # 5. IoU 계산 (0으로 나누기 방지)
    if union_area == 0:
        return 0.0

    iou = inter_area / union_area if union_area > 0 else 0
    return iou

# 테스트
box1 = [50, 50, 150, 150]
box2 = [100, 100, 200, 200]
print(f"Calculated IoU: {compute_iou(box1, box2):.4f}")

Calculated IoU: 0.1429


### 문제 4. NMS (Non-Maximum Suppression) 알고리즘 로직
객체 검출 결과로 나온 여러 박스 중 중복을 제거하는 NMS 알고리즘의 핵심 로직을 구현하시오.

(가장 스코어가 높은 박스를 기준으로 IoU가 특정 임계값 이상인 박스들을 제거합니다.)

In [4]:
def nms_logic(boxes, scores, iou_threshold=0.5):
    # 점수 기준으로 내림차순 정렬된 인덱스
    sorted_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    keep = []

    while len(sorted_indices) > 0:
        # 1. 가장 점수가 높은 박스 선택
        current_idx = sorted_indices.pop(0)
        keep.append(current_idx)

        remaining_indices = []
        for idx in sorted_indices:
            # 2. 선택된 박스와 나머지 박스들 간의 IoU 계산
            iou = compute_iou(boxes[current_idx], boxes[idx])

            # TODO: IoU가 임계값(iou_threshold)보다 작은 경우에만 남김 (중복이 아닌 것만 보존)
            if iou < iou_threshold:
                # TODO: remaining_indices에 추가
                remaining_indices.append(idx)

        # 남은 인덱스로 업데이트
        sorted_indices = remaining_indices

    return keep

### 문제 5. Custom Dataset: COCO 포맷 변환
COCO 데이터셋 형식은 `[x_min, y_min, width, height]`이지만, PyTorch의 Faster R-CNN 모델은 `[x_min, y_min, x_max, y_max]` 형식을 요구합니다. `__getitem__` 메서드 내에서 이 좌표 변환을 수행하는 코드를 작성하시오.

In [ ]:
import torch

def convert_coco_to_xyxy(coco_bbox):
    # TODO: x1, y1, x2, y2 계산
    x1 = x
    y1 = y
    x2 = x + w
    y2 = y + h

    return [x1, y1, x2, y2]

# 테스트
sample_coco_box = [100, 100, 50, 80] # x=100, y=100, w=50, h=80
print(f"Converted Box: {convert_coco_to_xyxy(sample_coco_box)}")

### 문제 6. Fine-tuning을 위한 모델 Head 교체
TorchVision에서 제공하는 사전 학습된 `fasterrcnn_resnet50_fpn` 모델을 로드한 후, 사용자의 데이터셋 클래스 수(num_classes)에 맞게 **Box Predictor** (분류기 및 좌표 회귀기)를 교체하는 코드를 완성하시오.

In [7]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

def get_custom_model(num_classes):
    # 1. COCO로 사전 학습된 모델 로드
    model = fasterrcnn_resnet50_fpn(weights='DEFAULT')

    # 2. 기존 분류기(Head)의 입력 특징 차원 수 가져오기
    in_features = model.roi_heads.box_predictor.cls_score.in_features

    # 3. 새로운 Head로 교체 (num_classes에 맞게)
    # TODO: FastRCNNPredictor를 사용하여 model.roi_heads.box_predictor를 재정의
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    return model

# 테스트 (배경 포함 클래스 수 3개 가정)
model = get_custom_model(num_classes=3)
print(model.roi_heads.box_predictor)

FastRCNNPredictor(
  (cls_score): Linear(in_features=1024, out_features=3, bias=True)
  (bbox_pred): Linear(in_features=1024, out_features=12, bias=True)
)
